In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",  # 用于下载函数
    "torch",
    "tokenizers"
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.2.0
torch version: 2.14.0
tokenizers version: 0.23.2


In [2]:
from reasoning_from_scratch.qwen3 import Qwen3Model, Qwen3Tokenizer

In [ ]:
import torch
import torch.nn as nn

class RMSNorm(nn.Module):
    #eps是防止除以零的最小常数 qwen3_compatible是混合精度计算保护开关
    def __init__(
            self,emb_dim,eps = 1e-6,bias = False,qwen3_compatible = True
    ):
        super.__init__()
        self.eps = eps
        self.qwen3_compatible = qwen3_compatible
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim)) if bias else None

    def forward(self,x):
        input_dtype = x.dtype

        if self.qwen3_compatible:
            #防止float在16位的时候进行公式计算导致精度溢出,在混合精度下提高到32位
            #numpy用astype修改类型
            #pytorch可以用to移动设备,也可以用to修改类型
            x = x.to(torch.float32)

        #计算分母
        variance = x.pow(2).mean(dim = -1,keepdim = True)
        #
        norm_x = x * torch.rsqrt(variance + self.eps)
        norm_x = norm_x * self.scale

        if self.shift is not None:
            norm_x = norm_x + self.shift

        #
        return norm_x.to(input_dtype)